# PathSnap: data processing and feature engineering
Run from the project root. This notebook inspects the 80 training and 10 student-test trips, builds candidate features, and exports weak labels separately. Reference route rows are not GPS-point labels.

The supplied raw files are unchanged. Processing removes flagged rows only from the exported cleaned observations.

In [2]:
# Project folders and shared functions.
from pathlib import Path
import pandas as pd
from pipeline import (
    Config, FEATURE_NAMES, RoadIndex, align_reference,
    build_candidate_features, clean_track, discover_trips,
    inventory, make_map, make_training_folds, plot_trip,
    process_dataset, read_graph, read_reference,
)

DATA = Path("Student_folder") / "Student folder"
RESULTS = Path("results")
SETTINGS = Config()


ModuleNotFoundError: No module named 'pyproj'

## Dataset inventory and quality checks
Check GPS validity, timestamps, graph endpoints, connectivity, route continuity and partition sizes.

In [ ]:
# Check all 90 supplied development trips.
trip_inventory = inventory(DATA)
trip_inventory.to_csv(RESULTS / "dataset_inventory.csv", index=False)
display(trip_inventory.groupby("partition").size().rename("trips"))
display(trip_inventory[["partition", "raw_observations", "clean_observations",
                        "motion_observations_removed"]].groupby("partition").sum())
assert trip_inventory.status.eq("ok").all()


partition
student_test    10
train           80
Name: trips, dtype: int64

,raw_observations,clean_observations,motion_observations_removed
partition,,,
student_test,19160,19156,4
train,208417,207731,686


## Preprocessing and features
GPS timestamps are stably sorted and duplicate times removed. Coordinates use local UTM metres. Candidate arcs retain original trip-local row IDs. Features use distance, geometry, heading, speed, time gaps and node degrees. No reference data enters feature generation.

Cleaning now removes an observation if its time gap from the previous raw point exceeds 30 seconds or its apparent speed exceeds 60 m/s. The original row ID is retained for observations that remain.

In [ ]:
# Follow one example GPS trip through candidate feature creation.
trip = discover_trips(DATA, "train")[0]
gps, cleaning_report = clean_track(trip)
nodes, arcs = read_graph(trip)
road = RoadIndex(nodes, arcs, SETTINGS)
candidates, observations = build_candidate_features(gps, road, SETTINGS)

print("Removed GPS rows:", cleaning_report["motion_observations_removed"])
print("Candidate feature rows:", len(candidates["X"]))
display(pd.DataFrame(candidates["X"], columns=FEATURE_NAMES).head())


## Reference alignment
Monotone alignment derives weak observation labels from the supplied arc sequence. Residual thresholds identify usable labels. These are not independently verified pointwise ground truth.

In [ ]:
# The route file gives ordered arcs, not a label for each GPS row.
# This step estimates rough labels and reports how many are usable.
route = read_reference(trip, len(arcs))
labels = align_reference(gps, road, route, SETTINGS)
print("GPS points with usable rough labels:", int(labels.label_usable.sum()))
display(labels.head())


GPS points with usable rough labels: 1263


,observation_id,reference_arc_id,route_position,alignment_distance_m,nearest_reference_distance_m,label_usable,reference_connected,label_source
0,0,64973,1,0.722768,0.722768,True,True,monotone_reference_alignment_weak
1,1,64973,1,0.277461,0.277461,True,True,monotone_reference_alignment_weak
2,2,64973,1,1.839765,1.839765,True,True,monotone_reference_alignment_weak
3,3,64973,1,6.147324,6.147324,True,True,monotone_reference_alignment_weak
4,4,64973,1,2.652252,2.652252,True,True,monotone_reference_alignment_weak


## GPS and road map
Blue shows GPS observations, green the supplied reference, and grey the road network. Predicted routes will be added after model training.

In [ ]:
# Both maps use only the supplied GPS and road-network files.
html_map = make_map(trip, RESULTS / "maps" / "training_trip_00000001.html")
png_plot = plot_trip(trip, RESULTS / "figures" / "training_trip_00000001.png")
print("Interactive map:", RESULTS / "maps" / "training_trip_00000001.html")
print("Static plot:", png_plot)


## Export all permitted trips
Each trip receives candidates.npz, targets.npz, observations.csv.gz, labels.csv.gz and metadata.json. Candidate misses and weak-label coverage remain in the summary. No evaluator data is accessed.

In [ ]:
# Run the same feature and rough-label steps on every permitted trip.
summary = process_dataset(DATA, RESULTS / "processed", SETTINGS)
summary.to_csv(RESULTS / "processing_summary.csv", index=False)
display(summary.groupby("partition")[[
    "raw_observations", "clean_observations",
    "motion_observations_removed", "candidate_rows",
]].sum())


train: 10/80 trips processed


train: 20/80 trips processed


train: 30/80 trips processed


train: 40/80 trips processed


train: 50/80 trips processed


train: 60/80 trips processed


train: 70/80 trips processed


train: 80/80 trips processed


student_test: 10/10 trips processed


,raw_observations,clean_observations,motion_observations_removed,candidate_rows
partition,,,,
student_test,19160,19156,4,130903
train,208417,207731,686,1354225


## Trip-level cross-validation
All rows from a trip remain in one fold. Learned scaling must be fitted inside the training folds in the later model notebook.

In [ ]:
# Keep complete trips together in each later model-validation fold.
folds = make_training_folds(DATA, RESULTS / "training_folds.csv")
display(folds.groupby("validation_fold").size().rename("trips"))


Next stages: train and compare five models, perform Bayesian tuning, then implement HMM/Viterbi, API and UI. Pointwise metrics must be described as agreement with derived weak labels.